# Kuroshio Daily R5 — 轻量时序增强与残差预测资格实验

**目的**：诊断原 EOF-TCN-LSTM 与同参数量 LSTM 差距，在保留两分支、train-only EOF 和 LSTM 主干基础上，比较：

- `EOF-TCN-LSTM`：原 R4 模型，无改动，对照。
- `R5-Temporal`：原空间特征 TCN 编码器后追加轻量多尺度因果时间卷积。
- `R5-ResidualPC`：时序增强 + EOF-PC 均值回归残差输出，保持原PC-MSE损失。
- `R5-ResidualPhys`：再加入公里尺度辅助损失，隔离训练目标的贡献。
- `Matched-LSTM`：参数量匹配的简单 LSTM，不删减。

**程序合同**：先用两个历史内部验证期（2008、2016）进行机制资格实验，它们不是最终测试期；优化超参数只能由资格实验的 QA 集决定。没有取得预先设定的验证优势则自动阻止正式训练，不能通过降低 Gate、删强基线或读取测试结果选型。

通过后，执行严格年度滚动后报（2005、2009、2013、2017、2021、2024）；增加未参加 R4 主要结果展示的确认年份（2006、2010、2014、2018、2022、2023），二者在报告中明确区分。**此前已经查看过 R4 测试结果，因而该组属于迭代后的后报复核，不能称完全 untouched test。**

**注意**：上传文件只有源码和中英文图片，未提供 R4 逐样本评分 CSV，因此目前无法在这里重算精确误差、显著性或证明 R5 一定获胜。本 Notebook 必须在你本地冻结数据上执行训练才能得到结论。

In [ ]:
# ===== 0. 配置：先核实三个目录；改好后从头顺序运行 =====
from dataclasses import dataclass, asdict
from pathlib import Path
import os, json, math, gc, time, random, hashlib, warnings, sys
import numpy as np
import pandas as pd

@dataclass
class CFG:
    ROOT: str = '/mnt/d/KuroshioDaily1993_2025'
    DATASET_REL: str = 'final/dataset/daily_dual_branch_paths_1993_2025_FINAL.npz'
    HANDOFF_REL: str = 'final/reports/LONG_DAILY_1993_2025_HANDOFF.json'
    QC_INDEX_REL: str = 'final/reports/最终逐日路径索引.csv'
    MANUAL_QUEUE_REL: str = 'reports/07_人工复核队列.csv'
    MANUAL_METADATA_REL: str = 'manual_qc/metadata'
    OUTPUT_REL: str = 'R5_轻量时序残差优化'
    HISTORY: int = 180
    LEADS: tuple = (30, 60, 90)
    TEST_YEARS: tuple = (2005, 2009, 2013, 2017, 2021, 2024)
    PURGE_DAYS: int = 7
    MAX_INPUT_GAP: int = 2
    MAX_MISSING_FRAC: float = 0.03
    MIN_DATA_YEARS: int = 25
    MIN_ANNUAL_COVERAGE: float = 0.85
    MIN_TRAIN: int = 500
    MIN_VAL: int = 100
    MIN_TEST: int = 100
    DROP_BRANCH1_FIRST: int = 12  # 43->31; same old representation
    BRANCH2_MAX_LON: float = 145.0
    EOF_VARIANCE_TARGET: float = 0.99
    BATCH_SIZE: int = 32
    EPOCHS_B1: int = 70
    EPOCHS_B2: int = 70
    PATIENCE: int = 12
    LR: float = 0.001
    WEIGHT_DECAY: float = 0.0001
    DROPOUT: float = 0.30
    SEEDS: tuple = (20,21,22,23,24)
    AR_LAGS: tuple = (3, 7)
    AR_ALPHAS: tuple = (0.1, 1., 10.)
    CLIM_HALF_WINDOW_DAYS: int = 15
    CLIM_MIN_SAMPLES: int = 10
    BOOT_N: int = 1000
    BOOT_BLOCK_DAYS: int = 30
    FORCE_TRAIN: bool = False
    USE_AMP: bool = True
    RUN_FULL_TRAINING: bool = True   # 必须先通过 Cell 5 和 Cell 9 的资格Gate
    TRAIN_OTHER_HYBRIDS: bool = False # 下一轮网络结构比较可启用，不改baseline主表
    FIG_DPI: int = 300
    PILOT_TEST_YEARS: tuple = (2009, 2017)  # QA: 2008/2016, inner early-stop: 2007/2015
    PILOT_SEEDS: tuple = (20, 21)
    PILOT_EPOCHS: int = 24
    PILOT_PATIENCE: int = 7
    GATE_MIN_GAIN_KM: float = 0.50
    GATE_MAX_LEAD_REGRESSION_KM: float = 2.0
    ALLOW_UNQUALIFIED_FULL: bool = False  # 固定实验合同，失败则只出诊断，不进行正式长训
    CONFIRM_YEARS: tuple = (2006,2010,2014,2018,2022,2023)
    PHYS_LOSS_WEIGHT: float = 0.35
    PHYS_KM_SCALE: float = 50.0
    TEMPORAL_DILATIONS: tuple = (1, 7, 21)
    TRAIN_ORIGINAL_ETL: bool = True
    QUALIFY_ON_INNER_QA: bool = True

cfg=CFG(); ROOT=Path(cfg.ROOT); DS=ROOT/cfg.DATASET_REL; OUT=ROOT/cfg.OUTPUT_REL
DATA=OUT/'实验数据'; CKPT=OUT/'模型权重'; PRED=OUT/'逐样本预测'; REP=OUT/'报告'; FZH=OUT/'图表'/'中文'; FEN=OUT/'图表'/'英文'
for d in (DATA,CKPT,PRED,REP,FZH,FEN): d.mkdir(parents=True,exist_ok=True)
print(json.dumps(asdict(cfg),indent=2,ensure_ascii=False))
if not DS.exists(): raise FileNotFoundError(f'未发现真正的冻结数据：{DS}。请先运行人工QC notebook最后的冻结与数据构建单元。')

In [ ]:
# ===== 1. GPU与字体/依赖检查（不合格立即停止，避免白训练）=====
import importlib.util
required=('torch','scipy','sklearn','matplotlib','tqdm','pandas','numpy')
missing=[v for v in required if importlib.util.find_spec(v) is None]
if missing: raise RuntimeError('WSL环境缺依赖：'+', '.join(missing)+'；用当前Kernel对应python执行 python -m pip install torch scipy scikit-learn matplotlib tqdm pandas numpy')
import torch, torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from scipy.spatial import cKDTree
from sklearn.linear_model import Ridge
import matplotlib, matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.font_manager import FontProperties
from tqdm.auto import tqdm
print('kernel=',sys.executable,'torch=',torch.__version__,'CUDA=',torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('本轮RTX5080正式训练要求CUDA GPU；请检查WSL nvidia-smi 与 PyTorch CUDA构建。')
print('GPU=',torch.cuda.get_device_name(0)); DEV=torch.device('cuda')
torch.set_float32_matmul_precision('high')
font_paths=['/mnt/c/Windows/Fonts/msyh.ttc','/mnt/c/Windows/Fonts/simhei.ttf','/mnt/c/Windows/Fonts/simsun.ttc','/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc']
CJK_NAME=None
for path in font_paths:
    if Path(path).exists():
        try:
            font_manager.fontManager.addfont(path)
            CJK_NAME=FontProperties(fname=path).get_name(); break
        except Exception: pass
if CJK_NAME is None:
    raise RuntimeError('中文字体缺失：请在WSL安装 sudo apt install fonts-noto-cjk 或检查 Windows /mnt/c/Windows/Fonts，避免绘图乱码。')
print('Chinese font=',CJK_NAME)

In [ ]:
# ===== 2. 最终数据读取、年度覆盖、人工QC冻结核对 =====
with np.load(DS,allow_pickle=False) as z:
    dates=pd.DatetimeIndex(pd.to_datetime(z['dates'])).normalize()
    b1=np.asarray(z['branch1_lon'],np.float32)
    lat1=np.asarray(z['branch1_target_lats'],float)
    b2=np.asarray(z['branch2_lat'],np.float32)
    lon2=np.asarray(z['branch2_target_lons'],float)
    source=np.asarray(z['source']).astype(str) if 'source' in z.files else np.array(['unknown']*len(dates))
if not (len(dates)==len(b1)==len(b2)==len(source)): raise RuntimeError('dates / 两分支 / source 数量不一致')
if dates.has_duplicates: raise RuntimeError('发现重复日期，禁止将其直接用于训练')
ord=np.argsort(dates.values); dates=dates[ord]; b1=b1[ord];b2=b2[ord];source=source[ord]
if not np.isfinite(b1).all() or not np.isfinite(b2).all(): raise RuntimeError('冻结数据中含 NaN/Inf')

# 物理范围异常只作质控警示（不随意删除或修改路径），便于发现断面插值外推造成的不可信日
suspicious=(~((b1>=119)&(b1<=140)).all(axis=1)) | (~((b2>=15)&(b2<=50)).all(axis=1))
if suspicious.any():
    suspect_df=pd.DataFrame({'异常日期':dates[suspicious], '原因':'双分支坐标超出宽松物理范围，需人工核对'})
    suspect_df.to_csv(DATA/'01_待人工复查的物理范围异常路径.csv',index=False,encoding='utf-8-sig')
    warnings.warn(f'发现 {suspicious.sum()} 天路径越过宽松物理边界，建议正式训练前核实此文件')

if b1.shape[1]<=cfg.DROP_BRANCH1_FIRST: raise RuntimeError('分支1维数过小')
b1=b1[:,cfg.DROP_BRANCH1_FIRST:];lat1=lat1[cfg.DROP_BRANCH1_FIRST:]
keep2=lon2<=cfg.BRANCH2_MAX_LON+1e-8;b2=b2[:,keep2];lon2=lon2[keep2]
if len(b1)!=len(b2) or len(lat1)!=b1.shape[1] or len(lon2)!=b2.shape[1]: raise RuntimeError('双分支物理坐标不匹配')
if dates.min().year>1993 or dates.max().year<2024: raise RuntimeError('数据不覆盖1993–2024所需区间，不可宣称多年全历史验证')
cal=pd.date_range(dates.min(),dates.max(),freq='D'); pos=cal.get_indexer(dates)
valid=np.zeros(len(cal),bool);valid[pos]=True
A1=np.full((len(cal),b1.shape[1]),np.nan,np.float32);A2=np.full((len(cal),b2.shape[1]),np.nan,np.float32)
A1[pos]=b1; A2[pos]=b2
# causal forward fill (ONLY history, never target). All targets later read A1/A2 raw by exact date.
def causal_fill(raw,limit): return pd.DataFrame(raw).ffill(limit=limit).to_numpy(dtype=np.float32)
F1=causal_fill(A1,cfg.MAX_INPUT_GAP);F2=causal_fill(A2,cfg.MAX_INPUT_GAP)
annual=pd.DataFrame({'日期':cal,'有效':valid});annual['年份']=annual['日期'].dt.year
cover=annual.groupby('年份',as_index=False).agg(理论天数=('日期','size'),有效天数=('有效','sum'))
cover['有效比例']=cover['有效天数']/cover['理论天数'];cover.to_csv(DATA/'01_逐年覆盖率.csv',index=False,encoding='utf-8-sig')
if (cover['有效比例']>=cfg.MIN_ANNUAL_COVERAGE).sum()<cfg.MIN_DATA_YEARS:
    raise RuntimeError('具备≥85%有效覆盖的年份不足25年，请优先排查未完成人工修复或缺失年；详情见01_逐年覆盖率.csv')
hand=ROOT/cfg.HANDOFF_REL
if hand.exists():
    ho=json.loads(hand.read_text(encoding='utf-8'))
    print('HANDOFF final_daily_paths=',ho.get('final_daily_paths'))
    if int(ho.get('final_daily_paths',-1))!=len(dates): raise RuntimeError('HANDOFF与冻结npz样本数不一致')
else: warnings.warn('未发现HANDOFF.json；冻结数据可供资格检查，但交接记录尚未核实')
idxp=ROOT/cfg.QC_INDEX_REL
if idxp.exists():
    idx=pd.read_csv(idxp)
    if 'status' in idx.columns:
        unresolved=idx[~idx['status'].isin(['final','missing_raw','unusable'])]
        if len(unresolved): raise RuntimeError(f'最终逐日索引仍有 {len(unresolved)} 条未处理状态，停止训练')
queue=ROOT/cfg.MANUAL_QUEUE_REL; mdir=ROOT/cfg.MANUAL_METADATA_REL
if queue.exists():
    qq=pd.read_csv(queue)
    qdates=pd.to_datetime(qq['date']).dt.strftime('%Y%m%d') if len(qq) else []
    missing_meta=[q for q in qdates if not (mdir/f'{q}.json').exists()]
    if missing_meta: raise RuntimeError(f'{len(missing_meta)} 个manual复核队列日期尚无处理metadata: {missing_meta[:10]}')
    bad_meta=[]
    allowed={'manual_corrected','accepted_auto_after_manual_qc','unusable'}
    for q in qdates:
        try:
            state=json.loads((mdir/f'{q}.json').read_text(encoding='utf-8')).get('status')
            if state not in allowed:bad_meta.append((q,state))
        except Exception as e:bad_meta.append((q,repr(e)))
    if bad_meta:raise RuntimeError(f'{len(bad_meta)} 条人工QC仍不是已确认状态: {bad_meta[:10]}')
# CMEMS来源切换边界连续性检查：仅警告，不把真实大蛇行直接当成数据损坏
edge=pd.Timestamp('2025-01-01')
prev=np.flatnonzero(dates<edge)
post=np.flatnonzero(dates>=edge)
if len(prev) and len(post):
    i,j=int(prev[-1]),int(post[0])
    if (dates[j]-dates[i]).days<=5:
        d1=float(np.sqrt(np.mean((b1[j]-b1[i])**2)))
        d2=float(np.sqrt(np.mean((b2[j]-b2[i])**2)))
        pd.DataFrame([{'产品交界前':dates[i],'交界后':dates[j],
                       '分支1逐日经度RMSE_度':d1,'分支2逐日纬度RMSE_度':d2}]).to_csv(
            DATA/'01_2024到2025产品交界连续性.csv',index=False,encoding='utf-8-sig')
        if d1>1.0 or d2>1.5:
            warnings.warn('2024-2025 CMEMS来源交界发生较大路径跳变，应先检查跨产品物理一致性与流轴QC')
print('DATA PASS:',len(dates),'days;',dates.min().date(),'to',dates.max().date(),'; B1',b1.shape[1],'; B2',b2.shape[1])
display(cover)

In [ ]:
# ===== 3. 创建样本索引，不展开 [样本,180,特征] 巨型数组 =====
# target index = issue+lead; test targets require true valid day. input missing <=2 continuous days.
L=max(cfg.LEADS); H=cfg.HISTORY
missing=(~valid).astype(np.int32)
issue_indices=[];audit=[]
for i in range(H-1,len(cal)-L):
    if not valid[i]: continue
    win=missing[i-H+1:i+1]
    if win.mean()>cfg.MAX_MISSING_FRAC: continue
    if len(win) and np.max(np.diff(np.flatnonzero(np.r_[1,win==0,1]))-1)>cfg.MAX_INPUT_GAP: continue
    if any(not valid[i+d] for d in cfg.LEADS): continue
    if not (np.isfinite(F1[i-H+1:i+1]).all() and np.isfinite(F2[i-H+1:i+1]).all()): continue
    issue_indices.append(i)
ISS=np.asarray(issue_indices,np.int32)
if len(ISS)==0: raise RuntimeError('没有有效180→30/60/90 anchors，需检查缺日与冻结数据')
anchor=pd.DataFrame({'起报日期':cal[ISS]})
for l in cfg.LEADS: anchor[f'{l}天目标日期']=cal[ISS+l]
anchor.to_csv(DATA/'02_全部有效预测起报日期.csv',index=False,encoding='utf-8-sig')
print('anchor count=',len(ISS),' first=',cal[ISS[0]],'last=',cal[ISS[-1]])

In [ ]:
# ===== 4. 真正的 expanding-window 多年后报：test year = 2005/2009/2013/2017/2021/2024 =====
FOLDS=[];fr=[]; splitrows=[]
for fold_id,year in enumerate(cfg.TEST_YEARS,1):
    valstart=pd.Timestamp(year-1,1,1); teststart=pd.Timestamp(year,1,1)
    testend=pd.Timestamp(year,12,31)
    train_cut=valstart-pd.Timedelta(days=cfg.PURGE_DAYS)
    val_cut=teststart-pd.Timedelta(days=cfg.PURGE_DAYS)
    issues=cal[ISS]; latest=cal[ISS+L]
    tr=ISS[(latest<train_cut)]
    va=ISS[(issues>=valstart)&(issues<teststart)&(latest<val_cut)]
    te=ISS[(issues>=teststart)&(issues<=testend)]
    if len(tr)<cfg.MIN_TRAIN or len(va)<cfg.MIN_VAL or len(te)<cfg.MIN_TEST:
        raise RuntimeError(f'Fold {year} 样本不足: tr={len(tr)} val={len(va)} test={len(te)}；请查该年的日路径覆盖和连续缺日')
    if not(cal[tr+L].max()<cal[va].min()-pd.Timedelta(days=cfg.PURGE_DAYS) and cal[va+L].max()<cal[te].min()-pd.Timedelta(days=cfg.PURGE_DAYS)):
        raise RuntimeError(f'Fold {year} 发现严格时序泄漏')
    if (len(set(cal[tr+L])&set(cal[va+L])) or len(set(cal[tr+L])&set(cal[te+L])) or len(set(cal[va+L])&set(cal[te+L]))):
        raise RuntimeError(f'Fold {year} label date overlap')
    ff={'fold':fold_id,'year':year,'train':tr,'val':va,'test':te,'train_cut':train_cut,'val_cut':val_cut}
    FOLDS.append(ff)
    fr.append({'折次':fold_id,'测试年份':year,'训练样本':len(tr),'验证样本':len(va),'测试样本':len(te),
               '训练最晚标签日期':cal[tr+L].max(),'验证最早起报日期':cal[va].min(),
               '验证最晚标签日期':cal[va+L].max(),'测试最早起报日期':cal[te].min(),'测试最晚标签日期':cal[te+L].max(),'因果审计':'PASS'})
    for label,ix in [('训练',tr),('验证',va),('测试',te)]:
        splitrows.extend({'折次':fold_id,'测试年份':year,'集合':label,'起报日期':str(cal[i].date()),'最晚目标日期':str(cal[i+L].date())} for i in ix)
foldtab=pd.DataFrame(fr); foldtab.to_csv(DATA/'03_多年滚动后报折次及防泄漏审计.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(splitrows).to_csv(DATA/'03_逐样本时序划分.csv',index=False,encoding='utf-8-sig')
print('[STRICT MULTI-YEAR CAUSAL HINDCAST] ALL PASS'); display(foldtab)

In [ ]:
# ===== 5. 每折 train-only 标准化 + 自适应 99% EOF =====
# 使用该fold训练期的独立日记录，而不是重叠180d窗口重复叠加做SVD；内存恒定在 O(feature²)
class DailyEOF:
    def fit(self,obs,target=0.99):
        obs=np.asarray(obs,np.float64)
        self.mu=np.mean(obs,axis=0); self.sd=np.std(obs,axis=0); self.sd[self.sd<1e-6]=1.
        z=(obs-self.mu)/self.sd
        cov=(z.T@z)/max(1,len(z)-1)
        eig,vec=np.linalg.eigh(cov)
        idx=np.argsort(eig)[::-1];eig=np.maximum(eig[idx],0);vec=vec[:,idx]
        rat=np.cumsum(eig)/(eig.sum()+1e-12); n=int(np.searchsorted(rat,target)+1)
        self.V=np.asarray(vec[:,:n],np.float32);self.explained=float(rat[n-1]);self.n=n
        return self
    def pc(self,x): return (((np.asarray(x)-self.mu)/self.sd)@self.V).astype(np.float32)
    def invert(self,pc): return ((np.asarray(pc)@self.V.T)*self.sd+self.mu).astype(np.float32)
    def z(self,x): return ((np.asarray(x)-self.mu)/self.sd).astype(np.float32)

def make_fold_data(f):
    train_days=(cal< f['train_cut']) & valid
    pp1=DailyEOF().fit(A1[train_days],cfg.EOF_VARIANCE_TARGET)
    pp2=DailyEOF().fit(A2[train_days],cfg.EOF_VARIANCE_TARGET)
    for pp in (pp1,pp2):
        if pp.explained<cfg.EOF_VARIANCE_TARGET-1e-6: raise RuntimeError('EOF 99%训练方差Gate未通过')
    # approx memory: ~17k days * 2*(31+277) floats; never expands 180 day windows
    z1=pp1.z(F1);z2=pp2.z(F2)
    pcs1=pp1.pc(F1);pcs2=pp2.pc(F2)
    ppath=CKPT/f'第{f["fold"]}折_{f["year"]}年'; ppath.mkdir(exist_ok=True,parents=True)
    np.savez_compressed(ppath/'训练期预处理参数.npz',
        b1_mu=pp1.mu,b1_sd=pp1.sd,b1_V=pp1.V,b2_mu=pp2.mu,b2_sd=pp2.sd,b2_V=pp2.V,
        branch1_lat=lat1,branch2_lon=lon2,train_label_cutoff=str(f['train_cut'].date()))
    return {'pp1':pp1,'pp2':pp2,'z1':z1,'z2':z2,'p1':pcs1,'p2':pcs2,'path':ppath}

# 只预审一折，防止全局重复保留大数组
_testFD=make_fold_data(FOLDS[0]);print('fold1 EOF modes:',_testFD['pp1'].n,_testFD['pp2'].n,'explained:',_testFD['pp1'].explained,_testFD['pp2'].explained)
del _testFD;gc.collect()
print('[TRAIN-ONLY EOF 99%] PASS')

In [ ]:
# ===== 6. 动态加载数据集，训练时每个batch只展开当前180天窗口 =====
class CausalWindowDataset(Dataset):
    def __init__(self,issues,scaled,pcs):
        self.issues=np.asarray(issues,np.int32);self.scaled=scaled;self.pcs=pcs
    def __len__(self): return len(self.issues)
    def __getitem__(self,j):
        i=int(self.issues[j]);z=self.scaled[i-H+1:i+1]
        dz=np.empty_like(z);dz[0]=0;dz[1:]=z[1:]-z[:-1]
        x=np.concatenate([z,dz],axis=-1)
        y=np.stack([self.pcs[i+d] for d in cfg.LEADS])
        return torch.from_numpy(x.copy()),torch.from_numpy(y.copy())

class StepEncoder(nn.Module):
    # 与旧原稿的per-step TCN/Conv编码保持一致：输入路径特征卷积，然后LSTM承担时间建模
    def __init__(self,feature,branch,kind='tcn'):
        super().__init__();self.kind=kind;self.drop=nn.Dropout(cfg.DROPOUT)
        if kind=='tcn':
            channels=16 if branch==1 else 32;kernel=3 if branch==1 else 5
            self.conv=nn.Conv1d(1,channels,kernel,dilation=2)
            self.bn=nn.BatchNorm1d(channels)
            conv_len=feature+2-2*(kernel-1)
            self.fc=nn.Linear(channels*conv_len,32 if branch==1 else 64)
            self.out_dim=self.fc.out_features
        elif kind=='conv':
            c1=16 if branch==1 else 32;c2=32 if branch==1 else 64
            self.c1=nn.Conv1d(1,c1,3,padding=1);self.c2=nn.Conv1d(c1,c2,3,padding=1)
            self.out_dim=c2*feature
        else: raise ValueError(kind)
    def forward(self,x):
        B,T,F=x.shape; z=x.reshape(B*T,1,F)
        if self.kind=='tcn':
            z=torch.nn.functional.pad(z,(2,0)) # 与R3原架构一致的左侧填充
            z=self.drop(torch.relu(self.bn(self.conv(z))))
            z=self.drop(torch.relu(self.fc(z.flatten(1))))
        else:
            z=self.drop(torch.relu(self.c1(z)));z=self.drop(torch.relu(self.c2(z)));z=z.flatten(1)
        return z.reshape(B,T,-1)

class Hybrid(nn.Module):
    def __init__(self,feature,pc_dim,branch,kind='tcn',recur='lstm'):
        super().__init__();self.enc=StepEncoder(feature,branch,kind)
        h=32 if branch==1 else 64
        R=nn.GRU if recur=='gru' else nn.LSTM
        self.rnn=R(self.enc.out_dim,h,batch_first=True)
        self.fc=nn.Linear(h,3*pc_dim);self.pc_dim=pc_dim
    def forward(self,x):
        h,_=self.rnn(self.enc(x));return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

class MatchedLSTM(nn.Module):
    def __init__(self,feature,pc_dim,hidden):
        super().__init__();self.rnn=nn.LSTM(feature,hidden,batch_first=True)
        self.fc=nn.Linear(hidden,3*pc_dim);self.pc_dim=pc_dim
    def forward(self,x):
        h,_=self.rnn(x);return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

def nparams(m): return sum(p.numel() for p in m.parameters())
def match_lstm(feature,pc_dim,branch):
    target=nparams(Hybrid(feature,pc_dim,branch))
    # Pytorch single-layer LSTM: 4h(feature+h+2) + 3pc*h + 3pc
    opts=range(8,1025)
    h=min(opts,key=lambda hh:abs(4*hh*(feature+hh+2)+(3*pc_dim)*(hh+1)-target))
    actual=4*h*(feature+h+2)+(3*pc_dim)*(h+1)
    if abs(actual-target)/target>0.08: raise RuntimeError('Matched-LSTM参数量相对差>8%，不满足公平基线')
    return h,target,actual

def make_net(name,branch,fd):
    feature=fd[f'z{branch}'].shape[1]*2;pdim=fd[f'pp{branch}'].n
    if name=='Matched-LSTM':
        h,_,_=match_lstm(feature,pdim,branch);return MatchedLSTM(feature,pdim,h)
    if name=='EOF-TCN-LSTM': return Hybrid(feature,pdim,branch,'tcn','lstm')
    if name=='EOF-Conv-LSTM': return Hybrid(feature,pdim,branch,'conv','lstm')
    if name=='EOF-Conv-GRU': return Hybrid(feature,pdim,branch,'conv','gru')
    if name=='EOF-TCN-GRU': return Hybrid(feature,pdim,branch,'tcn','gru')
    raise ValueError(name)

# ===== R5: 仅增加轻量时间混合与PC残差头，保持原LSTM主干 =====
class CausalTemporalResidual(nn.Module):
    def __init__(self,dim,dilations=(1,7,21),dropout=0.10):
        super().__init__()
        self.blocks=nn.ModuleList()
        self.norms=nn.ModuleList()
        self.gates=nn.ParameterList()
        for d in dilations:
            # depthwise temporal conv + pointwise; forward严格左填充，不能看到未来
            layer=nn.Sequential(nn.Conv1d(dim,dim,3,dilation=d,groups=dim,bias=False),
                                nn.Conv1d(dim,dim,1),nn.GELU(),nn.Dropout(dropout))
            self.blocks.append(layer)
            self.norms.append(nn.LayerNorm(dim))
            self.gates.append(nn.Parameter(torch.tensor(-2.0)))
        self.dilations=dilations
    def forward(self,x):
        # [B,T,C], output at time t 只受 <=t 影响
        for block,norm,gate,d in zip(self.blocks,self.norms,self.gates,self.dilations):
            inp=x.transpose(1,2)
            z=block(torch.nn.functional.pad(inp,(2*d,0))).transpose(1,2)
            x=norm(x+torch.sigmoid(gate)*z)
        return x

class R5Hybrid(nn.Module):
    def __init__(self,feature,pc_dim,branch,kind,projection):
        super().__init__()
        self.kind=kind
        self.raw_features=feature//2
        self.encoder=StepEncoder(feature,branch,'tcn')
        self.temporal=CausalTemporalResidual(self.encoder.out_dim,cfg.TEMPORAL_DILATIONS)
        hidden=32 if branch==1 else 64
        self.rnn=nn.LSTM(self.encoder.out_dim,hidden,batch_first=True)
        self.fc=nn.Linear(hidden,3*pc_dim)
        self.pc_dim=pc_dim
        self.register_buffer('proj_V',torch.as_tensor(projection,dtype=torch.float32))
        # 均值回归 / persistence 先验；通过训练更新，不含未来标签
        self.mean_revert_logit=nn.Parameter(torch.logit(torch.tensor([0.45,0.30,0.18])))
        if kind in ('R5-ResidualPC','R5-ResidualPhys'):
            nn.init.zeros_(self.fc.weight)
            nn.init.zeros_(self.fc.bias)
    def forward(self,x):
        h=self.temporal(self.encoder(x))
        out,_=self.rnn(h)
        delta=self.fc(out[:,-1]).reshape(len(x),3,self.pc_dim)
        if self.kind in ('R5-ResidualPC','R5-ResidualPhys'):
            # raw x = [z, first_difference], z为使用训练集统计量标准化的路径
            pc_last=x[:,-1,:self.raw_features]@self.proj_V
            alpha=torch.sigmoid(self.mean_revert_logit).view(1,3,1)
            return alpha*pc_last[:,None,:]+delta
        return delta

def make_r5_model(name,branch,fd):
    feat=fd[f'z{branch}'].shape[1]*2
    pdim=fd[f'pp{branch}'].n
    if name in ('R5-Temporal','R5-ResidualPC','R5-ResidualPhys'):
        return R5Hybrid(feat,pdim,branch,name,fd[f'pp{branch}'].V)
    if name in ('EOF-TCN-LSTM','Matched-LSTM'):
        return make_net(name,branch,fd)
    raise ValueError(name)

def model_loss(pred, target, fd, branch, name):
    # 原R4对照完整保留其PC-MSE；R5-ResidualPhys仅加入可微的公里尺度辅助损失
    orig=torch.nn.functional.mse_loss(pred,target)
    if name!='R5-ResidualPhys': return orig
    pp=fd[f'pp{branch}']
    V=torch.as_tensor(pp.V.T,device=pred.device,dtype=pred.dtype)
    sd=torch.as_tensor(pp.sd,device=pred.device,dtype=pred.dtype)
    delta=(pred-target)@V*sd
    if branch==1:
        factor=torch.as_tensor(111.32*np.cos(np.deg2rad(lat1)),device=pred.device,dtype=pred.dtype)
    else:
        factor=torch.tensor(111.32,device=pred.device,dtype=pred.dtype)
    physical=(delta*factor)/cfg.PHYS_KM_SCALE
    h=torch.nn.functional.smooth_l1_loss(physical,torch.zeros_like(physical),beta=1.0)
    # 用训练EOF系数方差尺度归一化PC损失，避免两个分支因量纲不同权重失衡
    return (1-cfg.PHYS_LOSS_WEIGHT)*orig + cfg.PHYS_LOSS_WEIGHT*h

In [ ]:
# ===== 7. 五种基线/评估的科学定义（季节态和递推AR-EOF仅使用训练日期）=====
def persistence_pred(ix):
    return np.repeat(A1[ix,None,:],3,axis=1).astype(np.float32),np.repeat(A2[ix,None,:],3,axis=1).astype(np.float32)

def climatology_model(raw,trainmask):
    ids=np.flatnonzero(trainmask);dates_train=cal[ids]
    doy=np.array([t.dayofyear for t in dates_train],int)
    clims=[];glob=np.mean(raw[ids],axis=0)
    for day in range(1,367):
        cyc=np.minimum(np.abs(doy-day),366-np.abs(doy-day))
        ids2=ids[cyc<=cfg.CLIM_HALF_WINDOW_DAYS]
        clims.append(np.mean(raw[ids2],axis=0) if len(ids2)>=cfg.CLIM_MIN_SAMPLES else glob)
    return np.asarray(clims,np.float32)

def fit_recursive_AR(pc,rawvalid,train_cut,p,alpha):
    end=cal.searchsorted(train_cut,side='left')
    # 不把缺失日上已有causal fill的路径当成独立观测标签
    t=np.arange(p,end)
    ok=rawvalid[t].copy()
    for lag in range(1,p+1): ok &= rawvalid[t-lag]
    t=t[ok]
    if len(t)<200: raise RuntimeError('AR训练可用完整逐日连续片段不足200')
    XX=np.stack([pc[t-i] for i in range(p,0,-1)],axis=1).reshape(len(t),-1)
    yy=pc[t]
    reg=Ridge(alpha=alpha).fit(XX,yy)
    train_sd=np.std(pc[:end][rawvalid[:end]],axis=0)
    clip=5*np.maximum(train_sd,0.5)
    return reg,clip

def recursive_forecast(reg,clip,pc,issue,p):
    hist=np.asarray(pc[issue-p+1:issue+1],np.float32)
    if len(hist)!=p or not np.isfinite(hist).all(): raise RuntimeError('AR history bad')
    outs=[]
    for t in range(1,91):
        nxt=reg.predict(hist.reshape(1,-1))[0]
        nxt=np.clip(nxt,-clip,clip).astype(np.float32)
        hist=np.vstack((hist[1:],nxt))
        if t in cfg.LEADS: outs.append(nxt.copy())
    return np.stack(outs)

def tune_AR(fd,f,branch):
    pc=fd[f'p{branch}']; pp=fd[f'pp{branch}']; train_cut=f['train_cut']
    best=None
    validx=f['val'][::max(1,len(f['val'])//70)]
    for p in cfg.AR_LAGS:
        for alpha in cfg.AR_ALPHAS:
            reg,clip=fit_recursive_AR(pc,valid,train_cut,p,alpha)
            pred=np.stack([recursive_forecast(reg,clip,pc,int(i),p) for i in validx]);truth=np.stack([[pc[i+d] for d in cfg.LEADS] for i in validx])
            err=float(np.mean((pred-truth)**2))
            if best is None or err<best['val_mse']: best={'p':p,'alpha':alpha,'reg':reg,'clip':clip,'val_mse':err}
    return best

In [ ]:
# ===== 8. 物理公里误差：局地投影+最近点相邻线段的双向正交路径距离 =====
# 对于仅122-145E的分支2，重建得到共同定义域；不是对每一点简单经纬度平方误差。
# 下方是双向point-to-polyline近似cross-track，使用KDTree近邻三顶点及其邻接segments。
def merge_path(branch1,branch2):
    a=np.asarray(branch1,float);b=np.asarray(branch2,float)
    candidates=np.flatnonzero(lon2>a[-1]);
    k=int(candidates[np.argmin((lon2[candidates]-a[-1])**2+(b[candidates]-lat1[-1])**2)]) if len(candidates) else len(lon2)
    xs=np.r_[a,lon2[k:]];ys=np.r_[lat1,b[k:]]
    good=np.isfinite(xs)&np.isfinite(ys)
    return xs[good],ys[good]

def one_direction(A,B):
    # A,B: 2d coordinates (km); approximate nearest polyline segment for each A point
    if len(A)<2 or len(B)<2: return np.array([np.nan])
    tree=cKDTree(B);_,knn=tree.query(A,k=min(3,len(B)))
    knn=np.asarray(knn);knn=knn[:,None] if knn.ndim==1 else knn
    ix=np.unique(np.clip(np.concatenate([knn.ravel()-1,knn.ravel()]),0,len(B)-2))
    # segment shortlist globally may grow, but polynomial per path length (~300) and is vectorized
    v=B[ix+1]-B[ix];w=A[:,None,:]-B[ix][None,:,:]
    den=np.maximum(np.sum(v*v,axis=1),1e-12)
    t=np.clip(np.sum(w*v[None,:,:],axis=2)/den[None,:],0,1)
    q=B[ix][None,:,:]+t[:,:,None]*v[None,:,:]
    d=np.sqrt(np.sum((A[:,None,:]-q)**2,axis=2))
    return d.min(axis=1)

def errors_for_pair(pred1,pred2,truth1,truth2):
    x,y=merge_path(pred1,pred2);xt,yt=merge_path(truth1,truth2)
    lat0=np.mean(np.r_[y,yt]);lon0=np.mean(np.r_[x,xt]);c=np.cos(np.deg2rad(lat0))
    A=np.column_stack([(x-lon0)*111.32*c,(y-lat0)*111.32]);B=np.column_stack([(xt-lon0)*111.32*c,(yt-lat0)*111.32])
    d=np.r_[one_direction(A,B),one_direction(B,A)]
    return float(np.mean(d**2)),float(np.mean(d)),float(np.percentile(d,90))

def record_errors(fold,model,seed,issues,p1,p2):
    out=[]
    for n,i in enumerate(issues):
        for k,lead in enumerate(cfg.LEADS):
            y1=A1[i+lead];y2=A2[i+lead]
            mse,mean,p90=errors_for_pair(p1[n,k],p2[n,k],y1,y2)
            out.append({'折次':fold['fold'],'测试年份':fold['year'],'模型':model,'种子':seed,'起报日期':str(cal[i].date()),'目标日期':str(cal[i+lead].date()),'提前期_天':int(lead),
                        '横向路径误差MSE_km2':mse,'横向路径平均误差_km':mean,'横向路径P90_km':p90,
                        '分支1经度RMSE_度':float(np.sqrt(np.mean((p1[n,k]-y1)**2))),
                        '分支2纬度RMSE_度':float(np.sqrt(np.mean((p2[n,k]-y2)**2)))})
    return pd.DataFrame(out)

In [ ]:
# ===== 9. R5 训练封装、物理评价、可恢复 checkpoints =====
from contextlib import nullcontext

def r5_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def run_predict(net, issues, fd, branch):
    ds=CausalWindowDataset(issues,fd[f'z{branch}'],fd[f'p{branch}'])
    ld=DataLoader(ds,batch_size=cfg.BATCH_SIZE,shuffle=False,num_workers=0)
    arr=[];net.eval()
    with torch.no_grad():
        for xx,_ in ld: arr.append(net(xx.to(DEV)).float().cpu().numpy())
    return np.concatenate(arr,axis=0)

def fit_r5_branch(split, fd, name, seed, branch, root_dir, max_epochs=None, patience=None):
    max_epochs=cfg.PILOT_EPOCHS if max_epochs is None else max_epochs
    patience=cfg.PILOT_PATIENCE if patience is None else patience
    folder=root_dir/name/f'seed_{seed}'
    folder.mkdir(parents=True,exist_ok=True)
    weight=folder/f'branch{branch}_best.pt'; historyfile=folder/f'branch{branch}_history.csv'
    latest=folder/f'branch{branch}_latest.pt'
    # best checkpoint 附带实验合同签名；仅同一fold/样本划分/网络版本可复用
    contract={'name':name,'seed':seed,'branch':branch,'tr_n':int(len(split['train'])),
              'val_n':int(len(split['val'])),'tr_last':int(split['train'][-1]),
              'val_first':int(split['val'][0]),'max_epochs':max_epochs,
              'phys_weight':float(cfg.PHYS_LOSS_WEIGHT),'version':'R5-v1'}
    contract_id=hashlib.sha256(json.dumps(contract,sort_keys=True).encode()).hexdigest()
    r5_seed(seed)
    net=make_r5_model(name,branch,fd).to(DEV)
    if weight.exists() and historyfile.exists() and not cfg.FORCE_TRAIN:
        ck=torch.load(weight,map_location='cpu',weights_only=True)
        if ck.get('contract_id')==contract_id:
            net.load_state_dict(ck['state']);return net,pd.read_csv(historyfile)
    ds_tr=CausalWindowDataset(split['train'],fd[f'z{branch}'],fd[f'p{branch}'])
    ds_va=CausalWindowDataset(split['val'],fd[f'z{branch}'],fd[f'p{branch}'])
    ld_tr=DataLoader(ds_tr,batch_size=cfg.BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=True)
    ld_va=DataLoader(ds_va,batch_size=cfg.BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
    opt=torch.optim.AdamW(net.parameters(),lr=cfg.LR,weight_decay=cfg.WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,factor=0.5,patience=4)
    amp=bool(cfg.USE_AMP)
    scaler=torch.amp.GradScaler('cuda',enabled=amp)
    start_epoch=1;best=float('inf');bestep=0;bad=0;hist=[]
    if latest.exists() and not cfg.FORCE_TRAIN:
        try:
            ck=torch.load(latest,map_location='cpu',weights_only=True)
            if ck.get('contract_id')==contract_id:
                net.load_state_dict(ck['state']);opt.load_state_dict(ck['opt']);scheduler.load_state_dict(ck['scheduler']);scaler.load_state_dict(ck['scaler'])
                start_epoch=int(ck['epoch'])+1;best=float(ck['best']);bestep=int(ck['bestep']);bad=int(ck['bad']);hist=ck['hist']
                print('[RESUME]',folder,'epoch',start_epoch)
        except Exception as e:print('[resume invalid]',e)
    for ep in range(start_epoch,max_epochs+1):
        net.train();ts=tn=0
        for x,y in ld_tr:
            x=x.to(DEV,non_blocking=True);y=y.to(DEV,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',enabled=amp):
                pred=net(x)
                loss=model_loss(pred.float(),y.float(),fd,branch,name)
            if not torch.isfinite(loss):raise RuntimeError(f'{name} B{branch}: non-finite loss')
            scaler.scale(loss).backward();scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(),1.0)
            scaler.step(opt);scaler.update();ts+=float(loss.detach())*len(x);tn+=len(x)
        net.eval();vs=vn=0
        with torch.no_grad():
            for x,y in ld_va:
                x=x.to(DEV);y=y.to(DEV)
                with torch.autocast('cuda',enabled=amp):
                    p=net(x);loss=model_loss(p.float(),y.float(),fd,branch,name)
                vs+=float(loss.detach())*len(x);vn+=len(x)
        val=vs/max(1,vn);scheduler.step(val)
        hist.append({'epoch':ep,'train_loss':ts/max(1,tn),'val_loss':val,'lr':opt.param_groups[0]['lr']})
        if val<best-1e-7:
            best=val;bestep=ep;bad=0
            torch.save({'state':{k:v.detach().cpu() for k,v in net.state_dict().items()},'contract_id':contract_id,'epoch':ep,'best_val':best},weight)
        else:bad+=1
        if ep%5==0 or bad>=patience or ep==max_epochs:
            torch.save({'state':{k:v.detach().cpu() for k,v in net.state_dict().items()},'opt':opt.state_dict(),
                        'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'epoch':ep,'best':best,
                        'bestep':bestep,'bad':bad,'hist':hist,'contract_id':contract_id},latest)
        if ep==1 or ep%10==0 or bad>=patience:print(f'{name} B{branch} seed{seed} ep{ep} train={hist[-1]["train_loss"]:.5f} val={val:.5f}')
        if bad>=patience:break
    pd.DataFrame(hist).to_csv(historyfile,index=False,encoding='utf-8-sig')
    if latest.exists():latest.unlink()
    ck=torch.load(weight,map_location='cpu',weights_only=True)
    net.load_state_dict(ck['state']);net.eval()
    return net,pd.DataFrame(hist)

def evaluate_issues(f,name,seed,fd,issues,bmodel1,bmodel2):
    return record_errors(f,name,seed,issues,bmodel1,bmodel2)

def get_predictions(name,split,fd,seed,folder,max_epochs=None,patience=None):
    branches=[]
    for branch in (1,2):
        net,hist=fit_r5_branch(split,fd,name,seed,branch,folder,max_epochs,patience)
        pr=run_predict(net,split.get('qa',split.get('test')),fd,branch)
        branches.append(fd[f'pp{branch}'].invert(pr))
        del net,pr;torch.cuda.empty_cache()
    return branches

In [ ]:
# ===== 10. R5 架构 / 纯因果 / 梯度冒烟预检 =====
f=FOLDS[0];fd=make_fold_data(f)
smoke=[]
for name in ['Matched-LSTM','EOF-TCN-LSTM','R5-Temporal','R5-ResidualPC','R5-ResidualPhys']:
    for b in (1,2):
        m=make_r5_model(name,b,fd).to(DEV)
        ds=CausalWindowDataset(f['train'][:12],fd[f'z{b}'],fd[f'p{b}'])
        x,y=next(iter(DataLoader(ds,batch_size=3)))
        x=x.to(DEV);y=y.to(DEV)
        m.eval()
        with torch.no_grad():
            p=m(x)
            if tuple(p.shape)!=tuple(y.shape):raise RuntimeError(f'{name} shape mismatch {tuple(p.shape)} vs {tuple(y.shape)}')
            if name.startswith('R5-'):
                # 更改未来时刻的输入，早期hidden不得改变；检验TemporalMixer因果性
                z=m.encoder(x)
                prev=m.temporal(z)
                z2=z.clone();z2[:,90:]+=torch.randn_like(z2[:,90:])*0.5
                other=m.temporal(z2)
                causal_error=float((prev[:,:90]-other[:,:90]).abs().max())
                if causal_error>1e-5:raise RuntimeError(f'{name} temporal convolution leakage={causal_error}')
        m.train();p=m(x);loss=model_loss(p.float(),y.float(),fd,b,name)
        if not torch.isfinite(loss):raise RuntimeError(name+' nan loss')
        loss.backward()
        g=sum(v.grad is not None and torch.isfinite(v.grad).all() for v in m.parameters() if v.requires_grad)
        if g<3:raise RuntimeError(name+' insufficient gradients')
        smoke.append({'模型':name,'分支':b,'参数量':nparams(m),'loss':float(loss.detach()),'梯度层数':g,'结果':'PASS'})
        del m,x,y,p,loss;torch.cuda.empty_cache()
pd.DataFrame(smoke).to_csv(DATA/'R5_01_模型形状因果梯度预检.csv',index=False,encoding='utf-8-sig')
print('[R5 SHAPE+CAUSAL+GRAD PRECHECK] ALL PASS');display(pd.DataFrame(smoke))
del fd;gc.collect()

In [ ]:
# ===== 11. 内层早停与独立QA时间段：不能拿QA来早停或改训练损失 =====
def make_pilot_fold(f):
    year=f['year'];inner_year=year-2; qa_year=year-1
    qa_start=pd.Timestamp(qa_year,1,1)
    early_start=pd.Timestamp(inner_year,1,1)
    train_cut=early_start-pd.Timedelta(days=cfg.PURGE_DAYS)
    train=ISS[cal[ISS+L]<train_cut]
    # early-stop inner year issue，最后标签必须在QA起报之前
    early=ISS[(cal[ISS]>=early_start)&(cal[ISS]<qa_start)&(cal[ISS+L]<qa_start-pd.Timedelta(days=cfg.PURGE_DAYS))]
    qa=f['val']
    if len(train)<cfg.MIN_TRAIN or len(early)<cfg.MIN_VAL or len(qa)<cfg.MIN_VAL:
        raise RuntimeError(f'Pilot split {year} insufficient train={len(train)} early={len(early)} QA={len(qa)}')
    if not (cal[train+L].max()<cal[early].min()-pd.Timedelta(days=cfg.PURGE_DAYS) and
            cal[early+L].max()<cal[qa].min()-pd.Timedelta(days=cfg.PURGE_DAYS)):
        raise RuntimeError('Pilot temporal leakage')
    return {'fold':500+f['fold'],'year':f['year'],'train':train,'val':early,'qa':qa,'train_cut':train_cut}

PILOTS=[make_pilot_fold(next(f for f in FOLDS if f['year']==yr)) for yr in cfg.PILOT_TEST_YEARS]
pp=[]
for f in PILOTS:
    pp.append({'对应测试年份':f['year'],'训练样本':len(f['train']),'内层早停样本':len(f['val']),
               '独立机制资格样本':len(f['qa']), '训练最晚标签':str(cal[f['train']+L].max().date()),
               '早停最晚标签':str(cal[f['val']+L].max().date()),'QA最早起报':str(cal[f['qa']].min().date())})
pd.DataFrame(pp).to_csv(DATA/'R5_02_机制资格实验严格时间划分.csv',index=False,encoding='utf-8-sig')
print('[PILOT STRICT TEMPORAL QA] PASS');display(pd.DataFrame(pp))

In [ ]:
# ===== 12. 资格实验：先测4个网络×2fold×2seeds；不触碰test labels =====
PILOT_MODELS=['Matched-LSTM','EOF-TCN-LSTM','R5-Temporal','R5-ResidualPC','R5-ResidualPhys']
qa_csv=DATA/'R5_03_机制资格QA逐样本误差.csv'
qa_cache=OUT/'资格实验预测缓存';qa_cache.mkdir(parents=True,exist_ok=True)
rows=[]
for f in PILOTS:
    print('\n[PILOT]',f['year'],'QA year',f['year']-1)
    fd=make_fold_data(f)
    for name in PILOT_MODELS:
        for seed in cfg.PILOT_SEEDS:
            file=qa_cache/f'{f["year"]}_{name}_seed{seed}.npz'
            if file.exists() and not cfg.FORCE_TRAIN:
                with np.load(file) as z: a=z['b1'];b=z['b2']; ix=z['issues']
                if not np.array_equal(ix,f['qa']):raise RuntimeError(f'{file} QA indices mismatch')
            else:
                a,b=get_predictions(name,f,fd,seed,OUT/'资格实验权重',cfg.PILOT_EPOCHS,cfg.PILOT_PATIENCE)
                np.savez_compressed(file,b1=a,b2=b,issues=f['qa'])
            info=record_errors(f,name,seed,f['qa'],a,b)
            info['QA年份']=f['year']-1
            rows.append(info)
            pd.concat(rows,ignore_index=True).to_csv(qa_csv,index=False,encoding='utf-8-sig')
    del fd;gc.collect();torch.cuda.empty_cache()
QA=pd.concat(rows,ignore_index=True)
print('[PILOT DONE]',len(QA),'rows')

In [ ]:
# ===== 13. 按独立QA物理路径RMSE选机制，并实施固定资格门槛 =====
def aggregate_rmse(df):
    run=df.groupby(['QA年份','模型','种子','提前期_天'],as_index=False).agg(
        RMSE_km=('横向路径误差MSE_km2',lambda a:float(np.sqrt(np.mean(a)))))
    yr=run.groupby(['QA年份','模型','提前期_天'],as_index=False)['RMSE_km'].mean()
    agg=yr.groupby(['模型','提前期_天'],as_index=False)['RMSE_km'].mean()
    return run,yr,agg
runqa,yearqa,qa_mean=aggregate_rmse(QA)
qa_mean.to_csv(DATA/'R5_04_机制资格QA总体误差.csv',index=False,encoding='utf-8-sig')
ref=qa_mean[qa_mean['模型']=='Matched-LSTM'].set_index('提前期_天')['RMSE_km']
orig=qa_mean[qa_mean['模型']=='EOF-TCN-LSTM'].set_index('提前期_天')['RMSE_km']
checks=[]
for name in ('R5-Temporal','R5-ResidualPC','R5-ResidualPhys'):
    z=qa_mean[qa_mean['模型']==name].set_index('提前期_天')['RMSE_km']
    gain=float((ref-z).mean())
    gain_vs_orig=float((orig-z).mean())
    max_reg=float((z-ref).max())
    # 提前期等权，拒绝通过单个提前期的极端取巧改善掩盖显著回退
    ok=(gain>=cfg.GATE_MIN_GAIN_KM and gain_vs_orig>0 and max_reg<=cfg.GATE_MAX_LEAD_REGRESSION_KM)
    checks.append({'候选':name,'相对MatchedLSTM改善_km':gain,'相对原ETL改善_km':gain_vs_orig,
                   '最大单提前期退化_km':max_reg,'通过机制资格':bool(ok)})
check_df=pd.DataFrame(checks).sort_values('相对MatchedLSTM改善_km',ascending=False)
check_df.to_csv(DATA/'R5_05_机制资格硬门槛.csv',index=False,encoding='utf-8-sig')
qualified=check_df[check_df['通过机制资格']]
SELECTED_R5=qualified.iloc[0]['候选'] if len(qualified) else None
qual_record={'qualified':SELECTED_R5 is not None,'selected':SELECTED_R5,
             'gate':{'minimum_mean_gain_km':cfg.GATE_MIN_GAIN_KM,
                     'maximum_single_lead_regression_km':cfg.GATE_MAX_LEAD_REGRESSION_KM},
             'QA_years':[int(y-1) for y in cfg.PILOT_TEST_YEARS]}
(REP/'R5_机制资格判定.json').write_text(json.dumps(qual_record,ensure_ascii=False,indent=2),encoding='utf-8')
display(qa_mean);display(check_df)
print('R5 QUALIFICATION:',SELECTED_R5 if SELECTED_R5 else 'FAILED: Do not run full training')

In [ ]:
# ===== R5 机制资格中英文图和逐年份对照：失败也要保存，并不会隐去负面结果 =====
yearqa.to_csv(DATA/'R5_04b_机制资格各年份误差.csv',index=False,encoding='utf-8-sig')
COL={'Matched-LSTM':'#80ABC2','EOF-TCN-LSTM':'#557F95','R5-Temporal':'#427DA8','R5-ResidualPC':'#6F96A8','R5-ResidualPhys':'#356C83'}
ZH={'Matched-LSTM':'同参数量LSTM','EOF-TCN-LSTM':'原EOF-TCN-LSTM','R5-Temporal':'R5时序增强','R5-ResidualPC':'R5残差PC','R5-ResidualPhys':'R5残差增强'}
for lang,dirout,stem in [('zh',FZH,'机制资格比较_中文'),('en',FEN,'Mechanism_Qualification_English')]:
    plt.rcdefaults()
    matplotlib.rcParams['font.family']=[CJK_NAME,'DejaVu Sans'] if lang=='zh' else ['DejaVu Serif']
    matplotlib.rcParams.update({'axes.grid':True,'grid.linestyle':':','xtick.direction':'in','ytick.direction':'in','pdf.fonttype':42,'axes.unicode_minus':False})
    fig,ax=plt.subplots(figsize=(9.0,5.5))
    for name in PILOT_MODELS:
        q=qa_mean[qa_mean['模型']==name].sort_values('提前期_天')
        ax.plot(q['提前期_天'],q['RMSE_km'],color=COL[name],marker='o',markerfacecolor='white',
                label=ZH[name] if lang=='zh' else name)
    ax.set_xticks(cfg.LEADS)
    ax.set_xlabel('预测提前期（天）' if lang=='zh' else 'Forecast lead (days)')
    ax.set_ylabel('路径RMSE（km）' if lang=='zh' else 'Axis RMSE (km)')
    ax.set_title('内部机制资格验证（非测试年份）' if lang=='zh' else 'Internal mechanism qualification (not test years)')
    ax.legend(ncol=2,frameon=False);fig.tight_layout()
    fig.savefig(dirout/(stem+'.png'),dpi=cfg.FIG_DPI,bbox_inches='tight')
    fig.savefig(dirout/(stem+'.pdf'),bbox_inches='tight');plt.close(fig)
print('Mechanism qualification plots saved; pass=',SELECTED_R5 is not None)


In [ ]:
# ===== 14. R5 正式多年滚动后报：包含四基线、原ETL、MatchedLSTM和通过资格的R5 =====
if not cfg.RUN_FULL_TRAINING:
    raise RuntimeError('RUN_FULL_TRAINING=False，按配置只进行机制资格验证。')
if SELECTED_R5 is None and not cfg.ALLOW_UNQUALIFIED_FULL:
    raise RuntimeError('机制QA未通过；正式长训阻断。请查看R5_05_机制资格硬门槛.csv，不允许用测试年份调参或挑结果。')
if SELECTED_R5 is None:
    raise RuntimeError('R5无合格候选；不执行全文实验。')
MODELS=['Matched-LSTM','EOF-TCN-LSTM',SELECTED_R5]
rows=[]
ALL_FOLDS=FOLDS.copy()
# 额外确认期均为未在R4主图中展示的年份；避免混称完全untouched
for year in cfg.CONFIRM_YEARS:
    if year in cfg.TEST_YEARS:continue
    valstart=pd.Timestamp(year-1,1,1);teststart=pd.Timestamp(year,1,1);testend=pd.Timestamp(year,12,31)
    cut1=valstart-pd.Timedelta(days=cfg.PURGE_DAYS);cut2=teststart-pd.Timedelta(days=cfg.PURGE_DAYS)
    tr=ISS[cal[ISS+L]<cut1]
    va=ISS[(cal[ISS]>=valstart)&(cal[ISS]<teststart)&(cal[ISS+L]<cut2)]
    te=ISS[(cal[ISS]>=teststart)&(cal[ISS]<=testend)]
    if len(tr)<cfg.MIN_TRAIN or len(va)<cfg.MIN_VAL or len(te)<cfg.MIN_TEST:
        raise RuntimeError(f'confirm {year}: samples too small')
    if not(cal[tr+L].max()<cal[va].min()-pd.Timedelta(days=cfg.PURGE_DAYS) and
           cal[va+L].max()<cal[te].min()-pd.Timedelta(days=cfg.PURGE_DAYS)):
        raise RuntimeError(f'confirm year {year} chronology failed')
    ALL_FOLDS.append({'fold':100+year,'year':year,'train':tr,'val':va,'test':te,'train_cut':cut1,'val_cut':cut2})
ALL_FOLDS=sorted(ALL_FOLDS,key=lambda q:q['year'])
full_split=pd.DataFrame([{'测试年':f['year'],'样本训练':len(f['train']),'样本验证':len(f['val']),
                          '样本测试':len(f['test']), '分组':'R4已查看年份' if f['year'] in cfg.TEST_YEARS else 'R5额外确认年份'} for f in ALL_FOLDS])
full_split.to_csv(DATA/'R5_06_正式与确认年份划分.csv',index=False,encoding='utf-8-sig')
print('[R5 MULTIYEAR CAUSAL GATE] PASS');display(full_split)

In [ ]:
# ===== 15. 正式训练/传统基线重新计算/逐run落盘 =====
PRED5=OUT/'正式逐样本预测';PRED5.mkdir(parents=True,exist_ok=True)
RUNS5=OUT/'正式模型权重';RUNS5.mkdir(parents=True,exist_ok=True)
metrics_files=[]
for f in ALL_FOLDS:
    print('\n'+'#'*80+'  TEST',f['year'])
    fd=make_fold_data(f);ix=f['test'];trainmask=(cal<f['train_cut'])&valid
    for name in ['Persistence','Climatology','AR-EOF']:
        outcsv=DATA/f'第{f["year"]}年_{name}_逐起报误差.csv'
        cache=PRED5/f'{f["year"]}_{name}.npz'
        if outcsv.exists() and cache.exists() and not cfg.FORCE_TRAIN:
            with np.load(cache) as z:
                if not np.array_equal(z['issues'],ix):raise RuntimeError('baseline cache issue mismatch')
            metrics_files.append(outcsv);continue
        if name=='Persistence':a,b=persistence_pred(ix)
        elif name=='Climatology':
            c1=climatology_model(A1,trainmask);c2=climatology_model(A2,trainmask)
            day_idx=np.array([[(cal[i+l].dayofyear-1) for l in cfg.LEADS] for i in ix])
            a,b=c1[day_idx],c2[day_idx]
        else:
            ar1=tune_AR(fd,f,1);ar2=tune_AR(fd,f,2)
            p1=np.stack([recursive_forecast(ar1['reg'],ar1['clip'],fd['p1'],int(i),ar1['p']) for i in tqdm(ix,desc='AR-B1')])
            p2=np.stack([recursive_forecast(ar2['reg'],ar2['clip'],fd['p2'],int(i),ar2['p']) for i in tqdm(ix,desc='AR-B2')])
            a,b=fd['pp1'].invert(p1),fd['pp2'].invert(p2)
        np.savez_compressed(cache,b1=a,b2=b,issues=ix)
        record_errors(f,name,-1,ix,a,b).to_csv(outcsv,index=False,encoding='utf-8-sig')
        metrics_files.append(outcsv)
    for name in MODELS:
        for seed in cfg.SEEDS:
            outcsv=DATA/f'第{f["year"]}年_{name}_种子{seed}_逐起报误差.csv'
            cache=PRED5/f'{f["year"]}_{name}_seed{seed}.npz'
            if outcsv.exists() and cache.exists() and not cfg.FORCE_TRAIN:
                with np.load(cache) as z:
                    if not np.array_equal(z['issues'],ix):raise RuntimeError('neural cache issue mismatch')
                metrics_files.append(outcsv);continue
            print('TRAIN',f['year'],name,seed)
            # 正式训练每折只用该折训练+验证；QA年份不被重新用于选择R5改法
            split={'train':f['train'],'val':f['val'],'test':ix}
            aa,bb=get_predictions(name,split,fd,seed,RUNS5/f'{f["year"]}',cfg.EPOCHS_B1,cfg.PATIENCE)
            np.savez_compressed(cache,b1=aa,b2=bb,issues=ix)
            record_errors(f,name,seed,ix,aa,bb).to_csv(outcsv,index=False,encoding='utf-8-sig')
            metrics_files.append(outcsv)
    df=pd.concat([pd.read_csv(p) for p in metrics_files],ignore_index=True)
    df.to_csv(DATA/'R5_07_所有逐样本误差_中途快照.csv',index=False,encoding='utf-8-sig')
    del fd;gc.collect();torch.cuda.empty_cache()
all_metrics=pd.concat([pd.read_csv(p) for p in metrics_files],ignore_index=True)
all_metrics.to_csv(DATA/'R5_07_所有逐样本误差.csv',index=False,encoding='utf-8-sig')
print('R5 FULL complete:',len(all_metrics),'rows')

In [ ]:
# ===== 16. 实验统计：逐年RMSE，跨年等权平均，保留强基线 =====
if 'all_metrics' not in globals(): all_metrics=pd.read_csv(DATA/'R5_07_所有逐样本误差.csv')
rr=all_metrics.groupby(['测试年份','模型','种子','提前期_天'],as_index=False).agg(
    RMSE_km=('横向路径误差MSE_km2',lambda v:float(np.sqrt(np.mean(v)))),
    MAE_km=('横向路径平均误差_km','mean'),样本数=('横向路径误差MSE_km2','size'))
rr.to_csv(DATA/'R5_08_逐年种子正式误差.csv',index=False,encoding='utf-8-sig')
yr=rr.groupby(['测试年份','模型','提前期_天'],as_index=False).agg(RMSE_km=('RMSE_km','mean'),MAE_km=('MAE_km','mean'))
yr['结果类别']=np.where(yr['测试年份'].isin(cfg.TEST_YEARS),'R4已查看年份','R5额外确认年份')
yr.to_csv(DATA/'R5_09_逐年正式模型误差.csv',index=False,encoding='utf-8-sig')
summary=yr.groupby(['结果类别','模型','提前期_天'],as_index=False).agg(
    年份等权RMSE_km=('RMSE_km','mean'),跨年份标准差_km=('RMSE_km','std'),年份数=('RMSE_km','size'))
summary.to_csv(DATA/'R5_10_论文正式多年份指标.csv',index=False,encoding='utf-8-sig')
# 所有模型均需保留：哪怕模型差于Climatology或LSTM，也照实报告
for cat in summary['结果类别'].unique():
    q=summary[summary['结果类别']==cat]
    for lead in cfg.LEADS:
        qq=q[q['提前期_天']==lead].sort_values('年份等权RMSE_km')
        print(cat,lead,'day winner:',qq.iloc[0]['模型'],'RMSE=',float(qq.iloc[0]['年份等权RMSE_km']))
display(summary.sort_values(['结果类别','提前期_天','年份等权RMSE_km']))

In [ ]:
# ===== 17. 年份层级 + 配对时间块 Bootstrap，不将每天/随机种子视为独立年 =====
def circular_idx(n,block,rng):
    bl=max(1,min(n,int(block)))
    starts=rng.integers(0,n,size=int(np.ceil(n/bl)))
    return np.concatenate([(s+np.arange(bl))%n for s in starts])[:n]

def make_year_matrix(df,year,lead,model):
    q=df[(df['测试年份']==year)&(df['提前期_天']==lead)&(df['模型']==model)]
    return q.pivot_table(index='种子',columns='起报日期',values='横向路径误差MSE_km2').sort_index(axis=1)

models=['Persistence','Climatology','AR-EOF','Matched-LSTM','EOF-TCN-LSTM',SELECTED_R5]
boot_rows=[];paired_rows=[]
for cat,years in [('R4已查看年份',list(cfg.TEST_YEARS)),('R5额外确认年份',list(cfg.CONFIRM_YEARS))]:
    for lead in cfg.LEADS:
        rg=np.random.default_rng(20261013+lead+(0 if cat=='R4已查看年份' else 400))
        mats={}
        for year in years:
            frames={m:make_year_matrix(all_metrics,year,lead,m) for m in models}
            common=set.intersection(*(set(frame.columns) for frame in frames.values()))
            if len(common)<20:raise RuntimeError(f'bootstrap year={year}, too few aligned issues')
            dates=sorted(common)
            mats[year]={m:(np.asarray(frame.index),frame.reindex(columns=dates).to_numpy(float)) for m,frame in frames.items()}
        reps=np.empty((cfg.BOOT_N,len(models)))
        for i in range(cfg.BOOT_N):
            foldpick=rg.choice(years,size=len(years),replace=True)
            vals={m:[] for m in models}
            for yy in foldpick:
                mp=mats[yy];n=next(iter(mp.values()))[1].shape[1]
                timeids=circular_idx(n,cfg.BOOT_BLOCK_DAYS,rg)
                seedbase=np.asarray(cfg.SEEDS)
                seedids=rg.integers(0,len(seedbase),size=len(seedbase))
                for m in models:
                    ids,arr=mp[m]
                    if len(ids)==1:sample=arr[:,timeids]
                    elif np.array_equal(ids,seedbase):sample=arr[seedids][:,timeids]
                    else:sample=arr[rg.integers(0,len(ids),len(ids))][:,timeids]
                    vals[m].append(float(np.sqrt(np.mean(sample,axis=1)).mean()))
            reps[i]=[np.mean(vals[m]) for m in models]
        for j,m in enumerate(models):
            boot_rows.append({'结果类别':cat,'模型':m,'提前期_天':lead,'95CI下限_km':float(np.percentile(reps[:,j],2.5)),
                              '95CI上限_km':float(np.percentile(reps[:,j],97.5))})
        ix=models.index(SELECTED_R5)
        for j,m in enumerate(models):
            if j==ix:continue
            delta=reps[:,ix]-reps[:,j]
            paired_rows.append({'结果类别':cat,'模型A':SELECTED_R5,'模型B':m,'提前期_天':lead,
                                '差异均值_km':float(delta.mean()),'95CI下限_km':float(np.percentile(delta,2.5)),
                                '95CI上限_km':float(np.percentile(delta,97.5)),'CI跨零':bool(np.percentile(delta,2.5)<=0<=np.percentile(delta,97.5))})
bootci=pd.DataFrame(boot_rows);pairci=pd.DataFrame(paired_rows)
bootci.to_csv(DATA/'R5_11_年份层级Bootstrap_95CI.csv',index=False,encoding='utf-8-sig')
pairci.to_csv(DATA/'R5_12_模型差异成对Bootstrap_95CI.csv',index=False,encoding='utf-8-sig')
display(pairci)

In [ ]:
# ===== 18. 原稿风格的审稿配图：中/英 300dpi PNG + PDF；不隐去任何基线 =====
COL={'Persistence':'#777777','Climatology':'#C88A68','AR-EOF':'#9F8EA4','Matched-LSTM':'#80ABC2',
     'EOF-TCN-LSTM':'#557F95','R5-Temporal':'#427DA8','R5-ResidualPC':'#6F96A8','R5-ResidualPhys':'#356C83'}
ZH={'Persistence':'持续性预测','Climatology':'季节气候态','AR-EOF':'自回归EOF','Matched-LSTM':'同参数LSTM',
    'EOF-TCN-LSTM':'原EOF-TCN-LSTM','R5-Temporal':'R5时序增强','R5-ResidualPC':'R5残差PC','R5-ResidualPhys':'R5残差物理增强'}

def plot_style(lang):
    plt.rcdefaults();matplotlib.rcParams['font.family']=[CJK_NAME,'DejaVu Sans'] if lang=='zh' else ['DejaVu Serif']
    matplotlib.rcParams.update({'figure.facecolor':'white','axes.facecolor':'white','axes.edgecolor':'#222222','axes.linewidth':0.9,
        'axes.grid':True,'grid.color':'#DDDDDD','grid.linestyle':':','grid.linewidth':0.6,'xtick.direction':'in','ytick.direction':'in',
        'xtick.top':True,'ytick.right':True,'axes.unicode_minus':False,'pdf.fonttype':42,'legend.frameon':False,'font.size':10})

def save_both(fun,stemzh,stemen):
    for lang,folder,name in [('zh',FZH,stemzh),('en',FEN,stemen)]:
        plot_style(lang);fig=fun(lang)
        fig.savefig(folder/(name+'.png'),dpi=cfg.FIG_DPI,bbox_inches='tight')
        fig.savefig(folder/(name+'.pdf'),bbox_inches='tight')
        plt.close(fig)

def qacurve(lang):
    fig,ax=plt.subplots(figsize=(9,5.2))
    for m in PILOT_MODELS:
        qq=qa_mean[qa_mean['模型']==m].sort_values('提前期_天')
        ax.plot(qq['提前期_天'],qq['RMSE_km'],color=COL[m],marker='o',markerfacecolor='white',label=ZH[m] if lang=='zh' else m)
    ax.set_xticks(cfg.LEADS);ax.set_xlabel('预测提前期（天）' if lang=='zh' else 'Forecast lead (days)')
    ax.set_ylabel('路径RMSE（km）' if lang=='zh' else 'Axis RMSE (km)')
    ax.set_title('内部独立QA机制资格比较' if lang=='zh' else 'Mechanism qualification on inner QA periods')
    ax.legend(ncol=2);fig.tight_layout();return fig
save_both(qacurve,'图1_轻量机制资格比较_中文','Figure1_Mechanism_Qualification_English')

def official_curve(cat):
    def ff(lang):
        fig,ax=plt.subplots(figsize=(9,5.5))
        for m in models:
            qq=summary[(summary['结果类别']==cat)&(summary['模型']==m)].sort_values('提前期_天')
            ax.plot(qq['提前期_天'],qq['年份等权RMSE_km'],marker='o',markerfacecolor='white',color=COL[m],label=ZH[m] if lang=='zh' else m)
        ax.set_xticks(cfg.LEADS);ax.set_xlabel('预测提前期（天）' if lang=='zh' else 'Forecast lead (days)')
        ax.set_ylabel('跨年路径RMSE（km）' if lang=='zh' else 'Across-year axis RMSE (km)')
        ax.set_title(cat if lang=='zh' else ('Previously examined years' if cat=='R4已查看年份' else 'Additional confirmation years'))
        ax.legend(ncol=2);fig.tight_layout();return fig
    return ff
save_both(official_curve('R4已查看年份'),'图2_原六年份全部基线比较_中文','Figure2_Original_Six_Years_All_Baselines_English')
save_both(official_curve('R5额外确认年份'),'图3_新增确认年份全部基线比较_中文','Figure3_Additional_Confirmation_Years_English')

def year90(lang):
    fig,ax=plt.subplots(figsize=(10,5.5))
    for m in models:
        q=yr[(yr['模型']==m)&(yr['提前期_天']==90)].sort_values('测试年份')
        ax.plot(q['测试年份'],q['RMSE_km'],marker='o',color=COL[m],label=ZH[m] if lang=='zh' else m)
    ax.set_xlabel('测试年份' if lang=='zh' else 'Hindcast year');ax.set_ylabel('90天轴线RMSE（km）' if lang=='zh' else '90-day axis RMSE (km)')
    ax.legend(ncol=2);fig.tight_layout();return fig
save_both(year90,'图4_逐年90天泛化误差_中文','Figure4_Yearwise_90Day_Generalization_English')

print('中文:',FZH,'\n英文:',FEN)

In [ ]:
# ===== 19. 复现与结果交接，附数据指纹、资格结果和结论局限 =====
sha=hashlib.sha256()
with open(DS,'rb') as fin:
    for block in iter(lambda:fin.read(8*1024*1024),b''):sha.update(block)
manifest5={'dataset':str(DS),'sha256':sha.hexdigest(),'daily_dates':len(dates),
           'experimental_version':'R5-v1','pilot_QA_years':[int(x-1) for x in cfg.PILOT_TEST_YEARS],
           'qualification_gate':qual_record,'selected_R5':SELECTED_R5,
           'original_R4_test_years':list(cfg.TEST_YEARS),'additional_confirmation_years':list(cfg.CONFIRM_YEARS),
           'all_baselines_retained':['Persistence','Climatology','AR-EOF','Matched-LSTM','EOF-TCN-LSTM'],
           'R5_changes':['lightweight causal temporal convolution','optional EOF-PC persistence/mean-reversion residual','optional differentiable km-oriented loss'],
           'selection':'pilot inner QA only; not based on official test labels',
           'caution':'R4 results already inspected; these are revised hindcasts, not blind final validation'}
(REP/'R5_正式实验交接.json').write_text(json.dumps(manifest5,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(manifest5,ensure_ascii=False,indent=2))

## 论文叙述约束

1. 严禁承诺、预设或者篡改“全面超越”。如果资格 Gate 失败，先归档诊断再讨论下一轮改进；不降低门槛绕过。
2. 所有 baseline 完整保留。公平 LSTM 依据原主干的参数量进行匹配；R5 新增参数量另行披露。
3. 正式实验保持相同数据、时间划分、预处理与指标；调参与架构选择仅由内部 QA 决定。
4. 独立流轴物理精度验证不由再分析派生路径预测实验替代。
5. 如果希望称为真正的 blind hold-out，需要新数据或预先封存从未用于模型开发的测试年份。